In [0]:
# ============================================================
# CELL: LOAD name.basics.tsv
# ============================================================

df_name_basics = spark.read \
    .option("header", "true") \
    .option("sep", "\t") \
    .option("inferSchema", "true") \
    .option("nullValue", "\\N") \
    .csv("/Volumes/workspace/imdb_dw/data/name.basics.tsv")

df_name_basics.show(5, truncate=False)


+---------+---------------+---------+---------+---------------------------------+---------------------------------------+
|nconst   |primaryName    |birthYear|deathYear|primaryProfession                |knownForTitles                         |
+---------+---------------+---------+---------+---------------------------------+---------------------------------------+
|nm0000001|Fred Astaire   |1899     |1987     |actor,miscellaneous,producer     |tt0072308,tt0050419,tt0027125,tt0025164|
|nm0000002|Lauren Bacall  |1924     |2014     |actress,miscellaneous,soundtrack |tt0037382,tt0075213,tt0038355,tt0117057|
|nm0000003|Brigitte Bardot|1934     |NULL     |actress,music_department,producer|tt0057345,tt0049189,tt0056404,tt0054452|
|nm0000004|John Belushi   |1949     |1982     |actor,writer,music_department    |tt0072562,tt0077975,tt0080455,tt0078723|
|nm0000005|Ingmar Bergman |1918     |2007     |writer,director,actor            |tt0050986,tt0069467,tt0050976,tt0083922|
+---------+-------------

In [0]:
# ============================================================
# CELL: DETAILED PROFILE - name.basics.tsv
# ============================================================

from pyspark.sql.functions import col, count, when, countDistinct, max as spark_max, min as spark_min, length, sum as spark_sum

# Row count
total_rows = df_name_basics.count()

print("=" * 70)
print("📊 DETAILED PROFILE: name.basics.tsv")
print("=" * 70)
print(f"\n📌 Total Rows: {total_rows:,}")
print(f"📌 Total Columns: {len(df_name_basics.columns)}")
print(f"📌 Columns: {df_name_basics.columns}")

# Schema
print("\n" + "=" * 70)
print("📋 SCHEMA")
print("=" * 70)
df_name_basics.printSchema()

# Profile each column
print("\n" + "=" * 70)
print("📋 COLUMN-WISE PROFILING")
print("=" * 70)

for column in df_name_basics.columns:
    print(f"\n🔹 Column: {column}")
    print("-" * 40)
    
    stats = df_name_basics.agg(
        count(col(column)).alias("non_null_count"),
        countDistinct(col(column)).alias("distinct_count"),
        spark_sum(when(col(column).isNull(), 1).otherwise(0)).alias("null_count"),
        spark_max(length(col(column).cast("string"))).alias("max_len"),
        spark_min(length(col(column).cast("string"))).alias("min_len")
    ).collect()[0]
    
    non_null = stats["non_null_count"]
    distinct = stats["distinct_count"]
    nulls = stats["null_count"]
    max_len = stats["max_len"]
    min_len = stats["min_len"]
    null_pct = (nulls / total_rows * 100) if total_rows > 0 else 0
    
    print(f"   Non-Null Count : {non_null:,}")
    print(f"   Null Count     : {nulls:,} ({null_pct:.2f}%)")
    print(f"   Distinct Values: {distinct:,}")
    print(f"   Min Length     : {min_len}")
    print(f"   Max Length     : {max_len}")

# Sample data
print("\n" + "=" * 70)
print("📌 SAMPLE DATA (First 5 rows)")
print("=" * 70)
df_name_basics.show(5, truncate=False)

print("\n✅ Profiling complete for name.basics.tsv")
print("\n📸 TAKE SCREENSHOT FOR YOUR DOCUMENTATION!")

📊 DETAILED PROFILE: name.basics.tsv

📌 Total Rows: 14,903,875
📌 Total Columns: 6
📌 Columns: ['nconst', 'primaryName', 'birthYear', 'deathYear', 'primaryProfession', 'knownForTitles']

📋 SCHEMA
root
 |-- nconst: string (nullable = true)
 |-- primaryName: string (nullable = true)
 |-- birthYear: integer (nullable = true)
 |-- deathYear: integer (nullable = true)
 |-- primaryProfession: string (nullable = true)
 |-- knownForTitles: string (nullable = true)


📋 COLUMN-WISE PROFILING

🔹 Column: nconst
----------------------------------------
   Non-Null Count : 14,903,875
   Null Count     : 0 (0.00%)
   Distinct Values: 14,903,875
   Min Length     : 9
   Max Length     : 10

🔹 Column: primaryName
----------------------------------------
   Non-Null Count : 14,903,809
   Null Count     : 66 (0.00%)
   Distinct Values: 11,380,852
   Min Length     : 1
   Max Length     : 105

🔹 Column: birthYear
----------------------------------------
   Non-Null Count : 659,878
   Null Count     : 14,243,

In [0]:
# ============================================================
# CELL: ADDITIONAL ANALYSIS - name.basics.tsv
# ============================================================
# Analyze comma-separated columns: primaryProfession, knownForTitles

from pyspark.sql.functions import explode, split, trim

# Primary Key Check
print("=" * 70)
print("🔑 PRIMARY KEY CHECK: nconst")
print("=" * 70)
nconst_total = df_name_basics.count()
nconst_distinct = df_name_basics.select("nconst").distinct().count()
print(f"Total rows: {nconst_total:,}")
print(f"Distinct nconst: {nconst_distinct:,}")
if nconst_total == nconst_distinct:
    print("✅ nconst is UNIQUE - Valid Primary Key")
else:
    print(f"❌ DUPLICATES FOUND: {nconst_total - nconst_distinct:,}")

# Analyze primaryProfession (comma-separated)
print("\n" + "=" * 70)
print("📋 PRIMARY PROFESSION ANALYSIS")
print("=" * 70)

df_professions = df_name_basics \
    .filter(col("primaryProfession").isNotNull()) \
    .select(explode(split(col("primaryProfession"), ",")).alias("profession")) \
    .select(trim(col("profession")).alias("profession"))

profession_count = df_professions.select("profession").distinct().count()
print(f"Total distinct professions: {profession_count}")

print("\nTop 20 Professions:")
df_professions.groupBy("profession").count().orderBy(col("count").desc()).show(20, truncate=False)

print("\n✅ Additional analysis complete for name.basics.tsv")

🔑 PRIMARY KEY CHECK: nconst
Total rows: 14,903,875
Distinct nconst: 14,903,875
✅ nconst is UNIQUE - Valid Primary Key

📋 PRIMARY PROFESSION ANALYSIS
Total distinct professions: 46

Top 20 Professions:
+---------------------+-------+
|profession           |count  |
+---------------------+-------+
|actor                |3422031|
|actress              |2067934|
|producer             |1312751|
|miscellaneous        |1182387|
|writer               |984603 |
|camera_department    |873104 |
|director             |805539 |
|production_department|587477 |
|art_department       |463830 |
|cinematographer      |431829 |
|sound_department     |424904 |
|editor               |395658 |
|composer             |375130 |
|music_department     |310617 |
|assistant_director   |284167 |
|visual_effects       |267612 |
|make_up_department   |248313 |
|animation_department |241574 |
|archive_footage      |222201 |
|production_manager   |203934 |
+---------------------+-------+
only showing top 20 rows

✅ Add

In [0]:
# ============================================================
# CELL 4: DATA QUALITY CHECKS - name.basics.tsv
# ============================================================
# PURPOSE: Identify data quality issues that may cause rows to be
#          dropped or quarantined during ETL
# REQUIREMENT: Assignment says "If any rows dropped explain why"
# ============================================================

from pyspark.sql.functions import col, when

print("=" * 70)
print("🔍 DATA QUALITY CHECKS - name.basics.tsv")
print("=" * 70)

total_rows = df_name_basics.count()

# Check 1: Null or empty nconst (Primary Key)
null_nconst = df_name_basics.filter(
    (col("nconst").isNull()) | (col("nconst") == "")
).count()
print(f"\n❌ NULL/Empty nconst (PK): {null_nconst}")

# Check 2: Null primaryName
null_names = df_name_basics.filter(col("primaryName").isNull()).count()
print(f"⚠️ NULL primaryName: {null_names} ({null_names/total_rows*100:.4f}%)")

# Check 3: Future birth years (data quality issue)
future_births = df_name_basics.filter(col("birthYear") > 2025).count()
print(f"⚠️ Future birthYear (>2025): {future_births}")

# Check 4: Very old birth years (before 1800 - likely data error)
old_births = df_name_basics.filter(
    (col("birthYear").isNotNull()) & (col("birthYear") < 1800)
).count()
print(f"⚠️ Old birthYear (<1800): {old_births}")

# Check 5: Death before birth (impossible)
invalid_death = df_name_basics.filter(
    (col("deathYear").isNotNull()) & 
    (col("birthYear").isNotNull()) & 
    (col("deathYear") < col("birthYear"))
).count()
print(f"❌ DeathYear < BirthYear: {invalid_death}")

# Check 6: Null primaryProfession
null_profession = df_name_basics.filter(col("primaryProfession").isNull()).count()
print(f"⚠️ NULL primaryProfession: {null_profession} ({null_profession/total_rows*100:.2f}%)")

# Check 7: Null knownForTitles
null_titles = df_name_basics.filter(col("knownForTitles").isNull()).count()
print(f"⚠️ NULL knownForTitles: {null_titles} ({null_titles/total_rows*100:.2f}%)")

# Summary
print("\n" + "=" * 70)
print("📋 DATA QUALITY SUMMARY - name.basics.tsv")
print("=" * 70)

critical_issues = null_nconst + invalid_death
warning_issues = null_names + future_births + old_births

print(f"\n📌 Total Rows: {total_rows:,}")
print(f"❌ Critical Issues (will drop): {critical_issues}")
print(f"⚠️ Warning Issues (review needed): {warning_issues}")
print(f"✅ Clean Rows: {total_rows - critical_issues:,}")
print(f"\n📊 Data Quality Score: {((total_rows - critical_issues) / total_rows * 100):.2f}%")

# Show sample of problematic records (if any)
if null_names > 0:
    print("\n" + "=" * 70)
    print("📌 SAMPLE: Records with NULL primaryName")
    print("=" * 70)
    df_name_basics.filter(col("primaryName").isNull()).show(5, truncate=False)

if invalid_death > 0:
    print("\n" + "=" * 70)
    print("📌 SAMPLE: Records with DeathYear < BirthYear")
    print("=" * 70)
    df_name_basics.filter(
        (col("deathYear").isNotNull()) & 
        (col("birthYear").isNotNull()) & 
        (col("deathYear") < col("birthYear"))
    ).show(5, truncate=False)

print("\n" + "=" * 70)
print("📋 ETL DECISIONS FOR name.basics.tsv")
print("=" * 70)
print("""
Based on Data Quality Checks:
1. NULL nconst       → DROP (cannot have null primary key)
2. NULL primaryName  → KEEP (only 66 rows, minor issue)
3. Future birthYear  → KEEP but FLAG for review
4. Old birthYear     → KEEP (could be valid historical data)
5. DeathYear < Birth → DROP or QUARANTINE (invalid data)
6. NULL profession   → KEEP (will not create bridge record)
7. NULL knownFor     → KEEP (will not create bridge record)
""")

print("\n✅ Data Quality Checks complete for name.basics.tsv")
print("\n📸 TAKE SCREENSHOT FOR YOUR DOCUMENTATION!")

🔍 DATA QUALITY CHECKS - name.basics.tsv

❌ NULL/Empty nconst (PK): 0
⚠️ NULL primaryName: 66 (0.0004%)
⚠️ Future birthYear (>2025): 0
⚠️ Old birthYear (<1800): 816
❌ DeathYear < BirthYear: 29
⚠️ NULL primaryProfession: 2968923 (19.92%)
⚠️ NULL knownForTitles: 1752062 (11.76%)

📋 DATA QUALITY SUMMARY - name.basics.tsv

📌 Total Rows: 14,903,875
❌ Critical Issues (will drop): 29
⚠️ Warning Issues (review needed): 882
✅ Clean Rows: 14,903,846

📊 Data Quality Score: 100.00%

📌 SAMPLE: Records with NULL primaryName
+----------+-----------+---------+---------+-----------------+--------------+
|nconst    |primaryName|birthYear|deathYear|primaryProfession|knownForTitles|
+----------+-----------+---------+---------+-----------------+--------------+
|nm10019610|NULL       |NULL     |NULL     |NULL             |NULL          |
|nm10040984|NULL       |NULL     |NULL     |NULL             |NULL          |
|nm10405838|NULL       |NULL     |NULL     |NULL             |NULL          |
|nm10730114|NULL 

In [0]:
# ============================================================
# CELL 1: LOAD title.basics.tsv
# ============================================================

df_title_basics = spark.read \
    .option("header", "true") \
    .option("sep", "\t") \
    .option("inferSchema", "true") \
    .option("nullValue", "\\N") \
    .csv("/Volumes/workspace/imdb_dw/data/title.basics.tsv")

df_title_basics.show(5, truncate=False)

+---------+---------+----------------------+----------------------+-------+---------+-------+--------------+------------------------+
|tconst   |titleType|primaryTitle          |originalTitle         |isAdult|startYear|endYear|runtimeMinutes|genres                  |
+---------+---------+----------------------+----------------------+-------+---------+-------+--------------+------------------------+
|tt0000001|short    |Carmencita            |Carmencita            |0      |1894     |NULL   |1             |Documentary,Short       |
|tt0000002|short    |Le clown et ses chiens|Le clown et ses chiens|0      |1892     |NULL   |5             |Animation,Short         |
|tt0000003|short    |Poor Pierrot          |Pauvre Pierrot        |0      |1892     |NULL   |5             |Animation,Comedy,Romance|
|tt0000004|short    |Un bon bock           |Un bon bock           |0      |1892     |NULL   |12            |Animation,Short         |
|tt0000005|short    |Blacksmith Scene      |Blacksmith Scene  

In [0]:
# ============================================================
# CELL 2: DETAILED PROFILE - title.basics.tsv
# ============================================================

from pyspark.sql.functions import col, count, when, countDistinct, max as spark_max, min as spark_min, length, sum as spark_sum

total_rows = df_title_basics.count()

print("=" * 70)
print("📊 DETAILED PROFILE: title.basics.tsv")
print("=" * 70)
print(f"\n📌 Total Rows: {total_rows:,}")
print(f"📌 Total Columns: {len(df_title_basics.columns)}")
print(f"📌 Columns: {df_title_basics.columns}")

# Schema
print("\n" + "=" * 70)
print("📋 SCHEMA")
print("=" * 70)
df_title_basics.printSchema()

# Profile each column
print("\n" + "=" * 70)
print("📋 COLUMN-WISE PROFILING")
print("=" * 70)

for column in df_title_basics.columns:
    print(f"\n🔹 Column: {column}")
    print("-" * 40)
    
    stats = df_title_basics.agg(
        count(col(column)).alias("non_null_count"),
        countDistinct(col(column)).alias("distinct_count"),
        spark_sum(when(col(column).isNull(), 1).otherwise(0)).alias("null_count"),
        spark_max(length(col(column).cast("string"))).alias("max_len"),
        spark_min(length(col(column).cast("string"))).alias("min_len")
    ).collect()[0]
    
    non_null = stats["non_null_count"]
    distinct = stats["distinct_count"]
    nulls = stats["null_count"]
    max_len = stats["max_len"]
    min_len = stats["min_len"]
    null_pct = (nulls / total_rows * 100) if total_rows > 0 else 0
    
    print(f"   Non-Null Count : {non_null:,}")
    print(f"   Null Count     : {nulls:,} ({null_pct:.2f}%)")
    print(f"   Distinct Values: {distinct:,}")
    print(f"   Min Length     : {min_len}")
    print(f"   Max Length     : {max_len}")

# Sample data
print("\n" + "=" * 70)
print("📌 SAMPLE DATA (First 5 rows)")
print("=" * 70)
df_title_basics.show(5, truncate=False)

print("\n✅ Profiling complete for title.basics.tsv")

📊 DETAILED PROFILE: title.basics.tsv

📌 Total Rows: 12,095,664
📌 Total Columns: 9
📌 Columns: ['tconst', 'titleType', 'primaryTitle', 'originalTitle', 'isAdult', 'startYear', 'endYear', 'runtimeMinutes', 'genres']

📋 SCHEMA
root
 |-- tconst: string (nullable = true)
 |-- titleType: string (nullable = true)
 |-- primaryTitle: string (nullable = true)
 |-- originalTitle: string (nullable = true)
 |-- isAdult: integer (nullable = true)
 |-- startYear: integer (nullable = true)
 |-- endYear: integer (nullable = true)
 |-- runtimeMinutes: string (nullable = true)
 |-- genres: string (nullable = true)


📋 COLUMN-WISE PROFILING

🔹 Column: tconst
----------------------------------------
   Non-Null Count : 12,095,664
   Null Count     : 0 (0.00%)
   Distinct Values: 12,095,664
   Min Length     : 9
   Max Length     : 10

🔹 Column: titleType
----------------------------------------
   Non-Null Count : 12,095,664
   Null Count     : 0 (0.00%)
   Distinct Values: 11
   Min Length     : 5
   Max L

In [0]:
# ============================================================
# CELL 3: KEY ANALYSIS + DATA QUALITY - title.basics.tsv
# ============================================================
# Combined: PK check, Genre analysis, TitleType analysis, DQ checks

from pyspark.sql.functions import col, explode, split, trim

total_rows = df_title_basics.count()

# Primary Key Check
print("=" * 70)
print("🔑 PRIMARY KEY CHECK: tconst")
print("=" * 70)
tconst_distinct = df_title_basics.select("tconst").distinct().count()
print(f"Total rows: {total_rows:,}")
print(f"Distinct tconst: {tconst_distinct:,}")
if total_rows == tconst_distinct:
    print("✅ tconst is UNIQUE - Valid Primary Key")
else:
    print(f"❌ DUPLICATES FOUND: {total_rows - tconst_distinct:,}")

# TitleType Analysis (for filtering movies vs TV shows)
print("\n" + "=" * 70)
print("📋 TITLE TYPE ANALYSIS")
print("=" * 70)
df_title_basics.groupBy("titleType").count().orderBy(col("count").desc()).show(15, truncate=False)

# isAdult Analysis
print("\n" + "=" * 70)
print("📋 IS ADULT ANALYSIS")
print("=" * 70)
df_title_basics.groupBy("isAdult").count().show()

# Genre Analysis (for DIM_GENRE)
print("\n" + "=" * 70)
print("📋 GENRE ANALYSIS (for DIM_GENRE)")
print("=" * 70)

df_genres = df_title_basics \
    .filter(col("genres").isNotNull()) \
    .select(explode(split(col("genres"), ",")).alias("genre")) \
    .select(trim(col("genre")).alias("genre"))

genre_count = df_genres.select("genre").distinct().count()
print(f"Total distinct genres: {genre_count}")

print("\nAll Genres:")
df_genres.groupBy("genre").count().orderBy(col("count").desc()).show(30, truncate=False)

# Data Quality Checks
print("\n" + "=" * 70)
print("🔍 DATA QUALITY CHECKS")
print("=" * 70)

null_tconst = df_title_basics.filter(col("tconst").isNull()).count()
null_title = df_title_basics.filter(col("primaryTitle").isNull()).count()
null_type = df_title_basics.filter(col("titleType").isNull()).count()
null_genres = df_title_basics.filter(col("genres").isNull()).count()

print(f"❌ NULL tconst (PK):     {null_tconst}")
print(f"⚠️ NULL primaryTitle:   {null_title}")
print(f"⚠️ NULL titleType:      {null_type}")
print(f"⚠️ NULL genres:         {null_genres} ({null_genres/total_rows*100:.2f}%)")

# Summary
print("\n" + "=" * 70)
print("📋 QUALITY SUMMARY")
print("=" * 70)
print(f"Total Rows:      {total_rows:,}")
print(f"Critical Issues: {null_tconst}")
print(f"Quality Score:   {((total_rows - null_tconst) / total_rows * 100):.2f}%")

print("\n✅ Analysis complete for title.basics.tsv")
print("\n📸 TAKE SCREENSHOT FOR YOUR DOCUMENTATION!")

🔑 PRIMARY KEY CHECK: tconst
Total rows: 12,095,664
Distinct tconst: 12,095,664
✅ tconst is UNIQUE - Valid Primary Key

📋 TITLE TYPE ANALYSIS
+------------+-------+
|titleType   |count  |
+------------+-------+
|tvEpisode   |9324110|
|short       |1096605|
|movie       |732747 |
|video       |318831 |
|tvSeries    |291147 |
|tvMovie     |153095 |
|tvMiniSeries|66521  |
|tvSpecial   |55789  |
|videoGame   |45996  |
|tvShort     |10822  |
|tvPilot     |1      |
+------------+-------+


📋 IS ADULT ANALYSIS
+-------+--------+
|isAdult|   count|
+-------+--------+
|      1|  394191|
|   2019|       3|
|      0|11700947|
|   2020|       2|
|   1981|      18|
|   2017|       1|
|   2023|       7|
|   2022|       1|
|   1983|      18|
|   1984|      27|
|   1982|      28|
|   1965|       1|
|   1975|      16|
|   1977|      20|
|   1969|      13|
|   1976|      16|
|   1979|      62|
|   1986|      15|
|   2024|       3|
|   1966|       5|
+-------+--------+
only showing top 20 rows

📋 GENRE AN

In [0]:
# ============================================================
# CELL 1: LOAD title.akas.tsv
# ============================================================

df_title_akas = spark.read \
    .option("header", "true") \
    .option("sep", "\t") \
    .option("inferSchema", "true") \
    .option("nullValue", "\\N") \
    .csv("/Volumes/workspace/imdb_dw/data/title.akas.tsv")

df_title_akas.show(5, truncate=False)

+---------+--------+-------------------------+------+--------+-----------+-------------+---------------+
|titleId  |ordering|title                    |region|language|types      |attributes   |isOriginalTitle|
+---------+--------+-------------------------+------+--------+-----------+-------------+---------------+
|tt0000001|1       |Carmencita               |NULL  |NULL    |original   |NULL         |1              |
|tt0000001|2       |Carmencita               |DE    |NULL    |NULL       |literal title|0              |
|tt0000001|3       |Carmencita               |US    |NULL    |imdbDisplay|NULL         |0              |
|tt0000001|4       |Carmencita - spanyol tánc|HU    |NULL    |imdbDisplay|NULL         |0              |
|tt0000001|5       |Καρμενσίτα               |GR    |NULL    |imdbDisplay|NULL         |0              |
+---------+--------+-------------------------+------+--------+-----------+-------------+---------------+
only showing top 5 rows


In [0]:
# ============================================================
# CELL 2: DETAILED PROFILE - title.akas.tsv
# ============================================================

from pyspark.sql.functions import col, count, when, countDistinct, max as spark_max, min as spark_min, length, sum as spark_sum

total_rows = df_title_akas.count()

print("=" * 70)
print("📊 DETAILED PROFILE: title.akas.tsv")
print("=" * 70)
print(f"\n📌 Total Rows: {total_rows:,}")
print(f"📌 Total Columns: {len(df_title_akas.columns)}")
print(f"📌 Columns: {df_title_akas.columns}")

# Schema
print("\n" + "=" * 70)
print("📋 SCHEMA")
print("=" * 70)
df_title_akas.printSchema()

# Profile each column
print("\n" + "=" * 70)
print("📋 COLUMN-WISE PROFILING")
print("=" * 70)

for column in df_title_akas.columns:
    print(f"\n🔹 Column: {column}")
    print("-" * 40)
    
    stats = df_title_akas.agg(
        count(col(column)).alias("non_null_count"),
        countDistinct(col(column)).alias("distinct_count"),
        spark_sum(when(col(column).isNull(), 1).otherwise(0)).alias("null_count"),
        spark_max(length(col(column).cast("string"))).alias("max_len"),
        spark_min(length(col(column).cast("string"))).alias("min_len")
    ).collect()[0]
    
    non_null = stats["non_null_count"]
    distinct = stats["distinct_count"]
    nulls = stats["null_count"]
    max_len = stats["max_len"]
    min_len = stats["min_len"]
    null_pct = (nulls / total_rows * 100) if total_rows > 0 else 0
    
    print(f"   Non-Null Count : {non_null:,}")
    print(f"   Null Count     : {nulls:,} ({null_pct:.2f}%)")
    print(f"   Distinct Values: {distinct:,}")
    print(f"   Min Length     : {min_len}")
    print(f"   Max Length     : {max_len}")

# Sample data
print("\n" + "=" * 70)
print("📌 SAMPLE DATA (First 5 rows)")
print("=" * 70)
df_title_akas.show(5, truncate=False)

print("\n✅ Profiling complete for title.akas.tsv")

📊 DETAILED PROFILE: title.akas.tsv

📌 Total Rows: 54,124,127
📌 Total Columns: 8
📌 Columns: ['titleId', 'ordering', 'title', 'region', 'language', 'types', 'attributes', 'isOriginalTitle']

📋 SCHEMA
root
 |-- titleId: string (nullable = true)
 |-- ordering: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- region: string (nullable = true)
 |-- language: string (nullable = true)
 |-- types: string (nullable = true)
 |-- attributes: string (nullable = true)
 |-- isOriginalTitle: integer (nullable = true)


📋 COLUMN-WISE PROFILING

🔹 Column: titleId
----------------------------------------
   Non-Null Count : 54,124,127
   Null Count     : 0 (0.00%)
   Distinct Values: 12,090,919
   Min Length     : 9
   Max Length     : 10

🔹 Column: ordering
----------------------------------------
   Non-Null Count : 54,124,127
   Null Count     : 0 (0.00%)
   Distinct Values: 251
   Min Length     : 1
   Max Length     : 3

🔹 Column: title
----------------------------------------
   N

In [0]:
# ============================================================
# CELL 3: KEY ANALYSIS + DATA QUALITY - title.akas.tsv
# ============================================================
# Analysis: Region (DIM_REGION), Language (DIM_LANGUAGE), DQ checks

from pyspark.sql.functions import col

total_rows = df_title_akas.count()

# Note: This file has composite key (titleId + ordering), not single PK
print("=" * 70)
print("🔑 KEY CHECK: titleId + ordering (Composite Key)")
print("=" * 70)
distinct_titles = df_title_akas.select("titleId").distinct().count()
print(f"Total rows: {total_rows:,}")
print(f"Distinct titleId: {distinct_titles:,}")
print(f"Avg alternate titles per movie: {total_rows / distinct_titles:.1f}")

# Region Analysis (for DIM_REGION)
print("\n" + "=" * 70)
print("📋 REGION ANALYSIS (for DIM_REGION)")
print("=" * 70)
region_count = df_title_akas.select("region").distinct().count()
print(f"Total distinct regions: {region_count}")
print("\nTop 30 Regions:")
df_title_akas.groupBy("region").count().orderBy(col("count").desc()).show(30, truncate=False)

# Language Analysis (for DIM_LANGUAGE)
print("\n" + "=" * 70)
print("📋 LANGUAGE ANALYSIS (for DIM_LANGUAGE)")
print("=" * 70)
language_count = df_title_akas.select("language").distinct().count()
print(f"Total distinct languages: {language_count}")
print("\nTop 30 Languages:")
df_title_akas.groupBy("language").count().orderBy(col("count").desc()).show(30, truncate=False)

# Types Analysis
print("\n" + "=" * 70)
print("📋 TYPES ANALYSIS")
print("=" * 70)
df_title_akas.groupBy("types").count().orderBy(col("count").desc()).show(20, truncate=False)

# isOriginalTitle Analysis
print("\n" + "=" * 70)
print("📋 IS ORIGINAL TITLE ANALYSIS")
print("=" * 70)
df_title_akas.groupBy("isOriginalTitle").count().show()

# Data Quality Checks
print("\n" + "=" * 70)
print("🔍 DATA QUALITY CHECKS")
print("=" * 70)

null_titleId = df_title_akas.filter(col("titleId").isNull()).count()
null_title = df_title_akas.filter(col("title").isNull()).count()
null_region = df_title_akas.filter(col("region").isNull()).count()
null_language = df_title_akas.filter(col("language").isNull()).count()

print(f"❌ NULL titleId:    {null_titleId}")
print(f"⚠️ NULL title:      {null_title}")
print(f"⚠️ NULL region:     {null_region} ({null_region/total_rows*100:.2f}%)")
print(f"⚠️ NULL language:   {null_language} ({null_language/total_rows*100:.2f}%)")

# Summary
print("\n" + "=" * 70)
print("📋 QUALITY SUMMARY")
print("=" * 70)
print(f"Total Rows:      {total_rows:,}")
print(f"Critical Issues: {null_titleId}")
print(f"Quality Score:   {((total_rows - null_titleId) / total_rows * 100):.2f}%")

print("\n✅ Analysis complete for title.akas.tsv")
print("\n📸 TAKE SCREENSHOT FOR YOUR DOCUMENTATION!")

🔑 KEY CHECK: titleId + ordering (Composite Key)
Total rows: 54,124,127
Distinct titleId: 12,090,919
Avg alternate titles per movie: 4.5

📋 REGION ANALYSIS (for DIM_REGION)
Total distinct regions: 252

Top 30 Regions:
+------+--------+
|region|count   |
+------+--------+
|NULL  |12175175|
|DE    |5319587 |
|JP    |5299693 |
|FR    |5281237 |
|IN    |5236869 |
|ES    |5194081 |
|IT    |5170040 |
|PT    |5078463 |
|US    |1690998 |
|GB    |539542  |
|CA    |309293  |
|XWW   |207566  |
|AU    |198828  |
|BR    |140189  |
|RU    |126922  |
|MX    |121729  |
|PL    |103438  |
|GR    |99390   |
|FI    |94058   |
|SE    |88937   |
|HU    |84098   |
|NL    |69838   |
|PH    |69433   |
|AR    |69348   |
|NO    |67888   |
|TR    |62620   |
|DK    |62300   |
|CN    |59475   |
|XWG   |53929   |
|HK    |49997   |
+------+--------+
only showing top 30 rows

📋 LANGUAGE ANALYSIS (for DIM_LANGUAGE)
Total distinct languages: 111

Top 30 Languages:
+--------+--------+
|language|count   |
+--------+-------

In [0]:
# ============================================================
# CELL 1: LOAD title.crew.tsv
# ============================================================

df_title_crew = spark.read \
    .option("header", "true") \
    .option("sep", "\t") \
    .option("inferSchema", "true") \
    .option("nullValue", "\\N") \
    .csv("/Volumes/workspace/imdb_dw/data/title.crew.tsv")

df_title_crew.show(5, truncate=False)

+---------+---------+---------+
|tconst   |directors|writers  |
+---------+---------+---------+
|tt0000001|nm0005690|NULL     |
|tt0000002|nm0721526|NULL     |
|tt0000003|nm0721526|nm0721526|
|tt0000004|nm0721526|NULL     |
|tt0000005|nm0005690|NULL     |
+---------+---------+---------+
only showing top 5 rows


In [0]:
# ============================================================
# CELL 2: DETAILED PROFILE + DATA QUALITY - title.crew.tsv
# ============================================================

from pyspark.sql.functions import col, count, when, countDistinct, max as spark_max, min as spark_min, length, sum as spark_sum

total_rows = df_title_crew.count()

print("=" * 70)
print("📊 DETAILED PROFILE: title.crew.tsv")
print("=" * 70)
print(f"\n📌 Total Rows: {total_rows:,}")
print(f"📌 Total Columns: {len(df_title_crew.columns)}")
print(f"📌 Columns: {df_title_crew.columns}")

# Schema
print("\n" + "=" * 70)
print("📋 SCHEMA")
print("=" * 70)
df_title_crew.printSchema()

# Profile each column
print("\n" + "=" * 70)
print("📋 COLUMN-WISE PROFILING")
print("=" * 70)

for column in df_title_crew.columns:
    print(f"\n🔹 Column: {column}")
    print("-" * 40)
    
    stats = df_title_crew.agg(
        count(col(column)).alias("non_null_count"),
        countDistinct(col(column)).alias("distinct_count"),
        spark_sum(when(col(column).isNull(), 1).otherwise(0)).alias("null_count"),
        spark_max(length(col(column).cast("string"))).alias("max_len"),
        spark_min(length(col(column).cast("string"))).alias("min_len")
    ).collect()[0]
    
    non_null = stats["non_null_count"]
    distinct = stats["distinct_count"]
    nulls = stats["null_count"]
    max_len = stats["max_len"]
    min_len = stats["min_len"]
    null_pct = (nulls / total_rows * 100) if total_rows > 0 else 0
    
    print(f"   Non-Null Count : {non_null:,}")
    print(f"   Null Count     : {nulls:,} ({null_pct:.2f}%)")
    print(f"   Distinct Values: {distinct:,}")
    print(f"   Min Length     : {min_len}")
    print(f"   Max Length     : {max_len}")

# Sample data
print("\n" + "=" * 70)
print("📌 SAMPLE DATA (First 5 rows)")
print("=" * 70)
df_title_crew.show(5, truncate=False)

# Primary Key Check
print("\n" + "=" * 70)
print("🔑 PRIMARY KEY CHECK: tconst")
print("=" * 70)
tconst_distinct = df_title_crew.select("tconst").distinct().count()
print(f"Total rows: {total_rows:,}")
print(f"Distinct tconst: {tconst_distinct:,}")
if total_rows == tconst_distinct:
    print("✅ tconst is UNIQUE - Valid Primary Key")
else:
    print(f"❌ DUPLICATES FOUND: {total_rows - tconst_distinct:,}")

# Data Quality Checks
print("\n" + "=" * 70)
print("🔍 DATA QUALITY CHECKS")
print("=" * 70)

null_tconst = df_title_crew.filter(col("tconst").isNull()).count()
null_directors = df_title_crew.filter(col("directors").isNull()).count()
null_writers = df_title_crew.filter(col("writers").isNull()).count()
both_null = df_title_crew.filter(
    (col("directors").isNull()) & (col("writers").isNull())
).count()

print(f"❌ NULL tconst:              {null_tconst}")
print(f"⚠️ NULL directors:          {null_directors} ({null_directors/total_rows*100:.2f}%)")
print(f"⚠️ NULL writers:            {null_writers} ({null_writers/total_rows*100:.2f}%)")
print(f"⚠️ Both directors & writers NULL: {both_null} ({both_null/total_rows*100:.2f}%)")

# Summary
print("\n" + "=" * 70)
print("📋 QUALITY SUMMARY")
print("=" * 70)
print(f"Total Rows:      {total_rows:,}")
print(f"Critical Issues: {null_tconst}")
print(f"Quality Score:   {((total_rows - null_tconst) / total_rows * 100):.2f}%")

print("\n✅ Profiling complete for title.crew.tsv")
print("\n📸 TAKE SCREENSHOT FOR YOUR DOCUMENTATION!")

📊 DETAILED PROFILE: title.crew.tsv

📌 Total Rows: 12,095,664
📌 Total Columns: 3
📌 Columns: ['tconst', 'directors', 'writers']

📋 SCHEMA
root
 |-- tconst: string (nullable = true)
 |-- directors: string (nullable = true)
 |-- writers: string (nullable = true)


📋 COLUMN-WISE PROFILING

🔹 Column: tconst
----------------------------------------
   Non-Null Count : 12,095,664
   Null Count     : 0 (0.00%)
   Distinct Values: 12,095,664
   Min Length     : 9
   Max Length     : 10

🔹 Column: directors
----------------------------------------
   Non-Null Count : 6,797,590
   Null Count     : 5,298,074 (43.80%)
   Distinct Values: 985,938
   Min Length     : 9
   Max Length     : 5371

🔹 Column: writers
----------------------------------------
   Non-Null Count : 6,110,251
   Null Count     : 5,985,413 (49.48%)
   Distinct Values: 1,465,705
   Min Length     : 9
   Max Length     : 14172

📌 SAMPLE DATA (First 5 rows)
+---------+---------+---------+
|tconst   |directors|writers  |
+---------+-

In [0]:
# ============================================================
# CELL 1: LOAD title.episode.tsv
# ============================================================

df_title_episode = spark.read \
    .option("header", "true") \
    .option("sep", "\t") \
    .option("inferSchema", "true") \
    .option("nullValue", "\\N") \
    .csv("/Volumes/workspace/imdb_dw/data/title.episode.tsv")

df_title_episode.show(5, truncate=False)

+---------+------------+------------+-------------+
|tconst   |parentTconst|seasonNumber|episodeNumber|
+---------+------------+------------+-------------+
|tt0031458|tt32857063  |NULL        |NULL         |
|tt0041951|tt0041038   |1           |9            |
|tt0042816|tt0989125   |1           |17           |
|tt0042889|tt0989125   |NULL        |NULL         |
|tt0043426|tt0040051   |3           |42           |
+---------+------------+------------+-------------+
only showing top 5 rows


In [0]:
# ============================================================
# CELL 2: DETAILED PROFILE + DATA QUALITY - title.episode.tsv
# ============================================================

from pyspark.sql.functions import col, count, when, countDistinct, max as spark_max, min as spark_min, length, sum as spark_sum

total_rows = df_title_episode.count()

print("=" * 70)
print("📊 DETAILED PROFILE: title.episode.tsv")
print("=" * 70)
print(f"\n📌 Total Rows: {total_rows:,}")
print(f"📌 Total Columns: {len(df_title_episode.columns)}")
print(f"📌 Columns: {df_title_episode.columns}")

# Schema
print("\n" + "=" * 70)
print("📋 SCHEMA")
print("=" * 70)
df_title_episode.printSchema()

# Profile each column
print("\n" + "=" * 70)
print("📋 COLUMN-WISE PROFILING")
print("=" * 70)

for column in df_title_episode.columns:
    print(f"\n🔹 Column: {column}")
    print("-" * 40)
    
    stats = df_title_episode.agg(
        count(col(column)).alias("non_null_count"),
        countDistinct(col(column)).alias("distinct_count"),
        spark_sum(when(col(column).isNull(), 1).otherwise(0)).alias("null_count"),
        spark_max(length(col(column).cast("string"))).alias("max_len"),
        spark_min(length(col(column).cast("string"))).alias("min_len")
    ).collect()[0]
    
    non_null = stats["non_null_count"]
    distinct = stats["distinct_count"]
    nulls = stats["null_count"]
    max_len = stats["max_len"]
    min_len = stats["min_len"]
    null_pct = (nulls / total_rows * 100) if total_rows > 0 else 0
    
    print(f"   Non-Null Count : {non_null:,}")
    print(f"   Null Count     : {nulls:,} ({null_pct:.2f}%)")
    print(f"   Distinct Values: {distinct:,}")
    print(f"   Min Length     : {min_len}")
    print(f"   Max Length     : {max_len}")

# Sample data
print("\n" + "=" * 70)
print("📌 SAMPLE DATA (First 5 rows)")
print("=" * 70)
df_title_episode.show(5, truncate=False)

# Primary Key Check
print("\n" + "=" * 70)
print("🔑 PRIMARY KEY CHECK: tconst (Episode ID)")
print("=" * 70)
tconst_distinct = df_title_episode.select("tconst").distinct().count()
print(f"Total rows: {total_rows:,}")
print(f"Distinct tconst: {tconst_distinct:,}")
if total_rows == tconst_distinct:
    print("✅ tconst is UNIQUE - Valid Primary Key")
else:
    print(f"❌ DUPLICATES FOUND: {total_rows - tconst_distinct:,}")

# Parent TV Series count
print("\n" + "=" * 70)
print("📋 PARENT TV SERIES ANALYSIS")
print("=" * 70)
parent_distinct = df_title_episode.select("parentTconst").distinct().count()
print(f"Distinct TV Series (parentTconst): {parent_distinct:,}")
print(f"Avg episodes per series: {total_rows / parent_distinct:.1f}")

# Data Quality Checks
print("\n" + "=" * 70)
print("🔍 DATA QUALITY CHECKS")
print("=" * 70)

null_tconst = df_title_episode.filter(col("tconst").isNull()).count()
null_parent = df_title_episode.filter(col("parentTconst").isNull()).count()
null_season = df_title_episode.filter(col("seasonNumber").isNull()).count()
null_episode = df_title_episode.filter(col("episodeNumber").isNull()).count()

print(f"❌ NULL tconst:        {null_tconst}")
print(f"❌ NULL parentTconst:  {null_parent}")
print(f"⚠️ NULL seasonNumber:  {null_season} ({null_season/total_rows*100:.2f}%)")
print(f"⚠️ NULL episodeNumber: {null_episode} ({null_episode/total_rows*100:.2f}%)")

# Summary
print("\n" + "=" * 70)
print("📋 QUALITY SUMMARY")
print("=" * 70)
critical = null_tconst + null_parent
print(f"Total Rows:      {total_rows:,}")
print(f"Critical Issues: {critical}")
print(f"Quality Score:   {((total_rows - critical) / total_rows * 100):.2f}%")

print("\n✅ Profiling complete for title.episode.tsv")
print("\n📸 TAKE SCREENSHOT FOR YOUR DOCUMENTATION!")

📊 DETAILED PROFILE: title.episode.tsv

📌 Total Rows: 9,324,110
📌 Total Columns: 4
📌 Columns: ['tconst', 'parentTconst', 'seasonNumber', 'episodeNumber']

📋 SCHEMA
root
 |-- tconst: string (nullable = true)
 |-- parentTconst: string (nullable = true)
 |-- seasonNumber: integer (nullable = true)
 |-- episodeNumber: integer (nullable = true)


📋 COLUMN-WISE PROFILING

🔹 Column: tconst
----------------------------------------
   Non-Null Count : 9,324,110
   Null Count     : 0 (0.00%)
   Distinct Values: 9,324,110
   Min Length     : 9
   Max Length     : 10

🔹 Column: parentTconst
----------------------------------------
   Non-Null Count : 9,324,110
   Null Count     : 0 (0.00%)
   Distinct Values: 228,980
   Min Length     : 9
   Max Length     : 10

🔹 Column: seasonNumber
----------------------------------------
   Non-Null Count : 7,422,661
   Null Count     : 1,901,449 (20.39%)
   Distinct Values: 328
   Min Length     : 1
   Max Length     : 4

🔹 Column: episodeNumber
--------------

In [0]:
# ============================================================
# CELL 1: LOAD title.principals.tsv
# ============================================================

df_title_principals = spark.read \
    .option("header", "true") \
    .option("sep", "\t") \
    .option("inferSchema", "true") \
    .option("nullValue", "\\N") \
    .csv("/Volumes/workspace/imdb_dw/data/title.principals.tsv")

df_title_principals.show(5, truncate=False)

+---------+--------+---------+---------------+-----------------------+----------+
|tconst   |ordering|nconst   |category       |job                    |characters|
+---------+--------+---------+---------------+-----------------------+----------+
|tt0000001|1       |nm1588970|self           |NULL                   |["Self"]  |
|tt0000001|2       |nm0005690|director       |NULL                   |NULL      |
|tt0000001|3       |nm0005690|producer       |producer               |NULL      |
|tt0000001|4       |nm0374658|cinematographer|director of photography|NULL      |
|tt0000002|1       |nm0721526|director       |NULL                   |NULL      |
+---------+--------+---------+---------------+-----------------------+----------+
only showing top 5 rows


In [0]:
# ============================================================
# CELL 2: DETAILED PROFILE - title.principals.tsv
# ============================================================

from pyspark.sql.functions import col, count, when, countDistinct, max as spark_max, min as spark_min, length, sum as spark_sum

total_rows = df_title_principals.count()

print("=" * 70)
print("📊 DETAILED PROFILE: title.principals.tsv")
print("=" * 70)
print(f"\n📌 Total Rows: {total_rows:,}")
print(f"📌 Total Columns: {len(df_title_principals.columns)}")
print(f"📌 Columns: {df_title_principals.columns}")

# Schema
print("\n" + "=" * 70)
print("📋 SCHEMA")
print("=" * 70)
df_title_principals.printSchema()

# Profile each column
print("\n" + "=" * 70)
print("📋 COLUMN-WISE PROFILING")
print("=" * 70)

for column in df_title_principals.columns:
    print(f"\n🔹 Column: {column}")
    print("-" * 40)
    
    stats = df_title_principals.agg(
        count(col(column)).alias("non_null_count"),
        countDistinct(col(column)).alias("distinct_count"),
        spark_sum(when(col(column).isNull(), 1).otherwise(0)).alias("null_count"),
        spark_max(length(col(column).cast("string"))).alias("max_len"),
        spark_min(length(col(column).cast("string"))).alias("min_len")
    ).collect()[0]
    
    non_null = stats["non_null_count"]
    distinct = stats["distinct_count"]
    nulls = stats["null_count"]
    max_len = stats["max_len"]
    min_len = stats["min_len"]
    null_pct = (nulls / total_rows * 100) if total_rows > 0 else 0
    
    print(f"   Non-Null Count : {non_null:,}")
    print(f"   Null Count     : {nulls:,} ({null_pct:.2f}%)")
    print(f"   Distinct Values: {distinct:,}")
    print(f"   Min Length     : {min_len}")
    print(f"   Max Length     : {max_len}")

# Sample data
print("\n" + "=" * 70)
print("📌 SAMPLE DATA (First 5 rows)")
print("=" * 70)
df_title_principals.show(5, truncate=False)

print("\n✅ Profiling complete for title.principals.tsv")

📊 DETAILED PROFILE: title.principals.tsv

📌 Total Rows: 96,132,477
📌 Total Columns: 6
📌 Columns: ['tconst', 'ordering', 'nconst', 'category', 'job', 'characters']

📋 SCHEMA
root
 |-- tconst: string (nullable = true)
 |-- ordering: integer (nullable = true)
 |-- nconst: string (nullable = true)
 |-- category: string (nullable = true)
 |-- job: string (nullable = true)
 |-- characters: string (nullable = true)


📋 COLUMN-WISE PROFILING

🔹 Column: tconst
----------------------------------------
   Non-Null Count : 96,132,477
   Null Count     : 0 (0.00%)
   Distinct Values: 10,929,831
   Min Length     : 9
   Max Length     : 10

🔹 Column: ordering
----------------------------------------
   Non-Null Count : 96,132,477
   Null Count     : 0 (0.00%)
   Distinct Values: 75
   Min Length     : 1
   Max Length     : 2

🔹 Column: nconst
----------------------------------------
   Non-Null Count : 96,132,477
   Null Count     : 0 (0.00%)
   Distinct Values: 6,891,831
   Min Length     : 9
   Ma

In [0]:
# ============================================================
# CELL 3: KEY ANALYSIS + DATA QUALITY - title.principals.tsv
# ============================================================
# Analysis: Category (crew types), Job analysis, DQ checks

from pyspark.sql.functions import col

total_rows = df_title_principals.count()

# Composite Key Check (tconst + ordering)
print("=" * 70)
print("🔑 KEY CHECK: tconst + ordering (Composite Key)")
print("=" * 70)
distinct_titles = df_title_principals.select("tconst").distinct().count()
print(f"Total rows: {total_rows:,}")
print(f"Distinct titles (tconst): {distinct_titles:,}")
print(f"Avg principals per title: {total_rows / distinct_titles:.1f}")

# Category Analysis (for FACT_TITLE_PRINCIPALS - crew/cast types)
print("\n" + "=" * 70)
print("📋 CATEGORY ANALYSIS (Crew/Cast Types)")
print("=" * 70)
print("This answers: 'Find all different types of crew/cast for a title'")
df_title_principals.groupBy("category").count().orderBy(col("count").desc()).show(20, truncate=False)

# Job Analysis
print("\n" + "=" * 70)
print("📋 JOB ANALYSIS (Top 30 Jobs)")
print("=" * 70)
job_count = df_title_principals.filter(col("job").isNotNull()).select("job").distinct().count()
print(f"Total distinct jobs: {job_count:,}")
print("\nTop 30 Jobs:")
df_title_principals.groupBy("job").count().orderBy(col("count").desc()).show(30, truncate=False)

# Data Quality Checks
print("\n" + "=" * 70)
print("🔍 DATA QUALITY CHECKS")
print("=" * 70)

null_tconst = df_title_principals.filter(col("tconst").isNull()).count()
null_nconst = df_title_principals.filter(col("nconst").isNull()).count()
null_category = df_title_principals.filter(col("category").isNull()).count()
null_job = df_title_principals.filter(col("job").isNull()).count()
null_characters = df_title_principals.filter(col("characters").isNull()).count()

print(f"❌ NULL tconst:     {null_tconst}")
print(f"❌ NULL nconst:     {null_nconst}")
print(f"⚠️ NULL category:   {null_category}")
print(f"⚠️ NULL job:        {null_job} ({null_job/total_rows*100:.2f}%)")
print(f"⚠️ NULL characters: {null_characters} ({null_characters/total_rows*100:.2f}%)")

# Summary
print("\n" + "=" * 70)
print("📋 QUALITY SUMMARY")
print("=" * 70)
critical = null_tconst + null_nconst
print(f"Total Rows:      {total_rows:,}")
print(f"Critical Issues: {critical}")
print(f"Quality Score:   {((total_rows - critical) / total_rows * 100):.2f}%")

print("\n✅ Analysis complete for title.principals.tsv")
print("\n📸 TAKE SCREENSHOT FOR YOUR DOCUMENTATION!")

🔑 KEY CHECK: tconst + ordering (Composite Key)
Total rows: 96,132,477
Distinct titles (tconst): 10,929,831
Avg principals per title: 8.8

📋 CATEGORY ANALYSIS (Crew/Cast Types)
This answers: 'Find all different types of crew/cast for a title'
+-------------------+--------+
|category           |count   |
+-------------------+--------+
|actor              |22797878|
|actress            |17219070|
|self               |14285116|
|writer             |11467734|
|director           |8236964 |
|producer           |7176186 |
|editor             |5068837 |
|cinematographer    |3895256 |
|composer           |3106790 |
|production_designer|1144626 |
|casting_director   |1115259 |
|archive_footage    |606257  |
|archive_sound      |12504   |
+-------------------+--------+


📋 JOB ANALYSIS (Top 30 Jobs)
Total distinct jobs: 45,869

Top 30 Jobs:
+-----------------------+--------+
|job                    |count   |
+-----------------------+--------+
|NULL                   |78106531|
|producer         

In [0]:
# ============================================================
# CELL 1: LOAD title.ratings.tsv
# ============================================================

df_title_ratings = spark.read \
    .option("header", "true") \
    .option("sep", "\t") \
    .option("inferSchema", "true") \
    .option("nullValue", "\\N") \
    .csv("/Volumes/workspace/imdb_dw/data/title.ratings.tsv")

df_title_ratings.show(5, truncate=False)

+---------+-------------+--------+
|tconst   |averageRating|numVotes|
+---------+-------------+--------+
|tt0000001|5.7          |2186    |
|tt0000002|5.5          |306     |
|tt0000003|6.4          |2270    |
|tt0000004|5.2          |196     |
|tt0000005|6.2          |3012    |
+---------+-------------+--------+
only showing top 5 rows


In [0]:
# ============================================================
# CELL 2: DETAILED PROFILE + DATA QUALITY - title.ratings.tsv
# ============================================================

from pyspark.sql.functions import col, count, when, countDistinct, max as spark_max, min as spark_min, length, sum as spark_sum, avg

total_rows = df_title_ratings.count()

print("=" * 70)
print("📊 DETAILED PROFILE: title.ratings.tsv")
print("=" * 70)
print(f"\n📌 Total Rows: {total_rows:,}")
print(f"📌 Total Columns: {len(df_title_ratings.columns)}")
print(f"📌 Columns: {df_title_ratings.columns}")

# Schema
print("\n" + "=" * 70)
print("📋 SCHEMA")
print("=" * 70)
df_title_ratings.printSchema()

# Profile each column
print("\n" + "=" * 70)
print("📋 COLUMN-WISE PROFILING")
print("=" * 70)

for column in df_title_ratings.columns:
    print(f"\n🔹 Column: {column}")
    print("-" * 40)
    
    stats = df_title_ratings.agg(
        count(col(column)).alias("non_null_count"),
        countDistinct(col(column)).alias("distinct_count"),
        spark_sum(when(col(column).isNull(), 1).otherwise(0)).alias("null_count"),
        spark_max(length(col(column).cast("string"))).alias("max_len"),
        spark_min(length(col(column).cast("string"))).alias("min_len")
    ).collect()[0]
    
    non_null = stats["non_null_count"]
    distinct = stats["distinct_count"]
    nulls = stats["null_count"]
    max_len = stats["max_len"]
    min_len = stats["min_len"]
    null_pct = (nulls / total_rows * 100) if total_rows > 0 else 0
    
    print(f"   Non-Null Count : {non_null:,}")
    print(f"   Null Count     : {nulls:,} ({null_pct:.2f}%)")
    print(f"   Distinct Values: {distinct:,}")
    print(f"   Min Length     : {min_len}")
    print(f"   Max Length     : {max_len}")

# Sample data
print("\n" + "=" * 70)
print("📌 SAMPLE DATA (First 5 rows)")
print("=" * 70)
df_title_ratings.show(5, truncate=False)

# Primary Key Check
print("\n" + "=" * 70)
print("🔑 PRIMARY KEY CHECK: tconst")
print("=" * 70)
tconst_distinct = df_title_ratings.select("tconst").distinct().count()
print(f"Total rows: {total_rows:,}")
print(f"Distinct tconst: {tconst_distinct:,}")
if total_rows == tconst_distinct:
    print("✅ tconst is UNIQUE - Valid Primary Key")
else:
    print(f"❌ DUPLICATES FOUND: {total_rows - tconst_distinct:,}")

# Rating Statistics
print("\n" + "=" * 70)
print("📋 RATING STATISTICS")
print("=" * 70)
rating_stats = df_title_ratings.agg(
    spark_min("averageRating").alias("min_rating"),
    spark_max("averageRating").alias("max_rating"),
    avg("averageRating").alias("avg_rating"),
    spark_min("numVotes").alias("min_votes"),
    spark_max("numVotes").alias("max_votes"),
    avg("numVotes").alias("avg_votes")
).collect()[0]

print(f"Average Rating: Min={rating_stats['min_rating']}, Max={rating_stats['max_rating']}, Avg={rating_stats['avg_rating']:.2f}")
print(f"Num Votes:      Min={rating_stats['min_votes']}, Max={rating_stats['max_votes']:,}, Avg={rating_stats['avg_votes']:.0f}")

# Data Quality Checks
print("\n" + "=" * 70)
print("🔍 DATA QUALITY CHECKS")
print("=" * 70)

null_tconst = df_title_ratings.filter(col("tconst").isNull()).count()
null_rating = df_title_ratings.filter(col("averageRating").isNull()).count()
null_votes = df_title_ratings.filter(col("numVotes").isNull()).count()

print(f"❌ NULL tconst:        {null_tconst}")
print(f"❌ NULL averageRating: {null_rating}")
print(f"❌ NULL numVotes:      {null_votes}")

# Summary
print("\n" + "=" * 70)
print("📋 QUALITY SUMMARY")
print("=" * 70)
critical = null_tconst + null_rating + null_votes
print(f"Total Rows:      {total_rows:,}")
print(f"Critical Issues: {critical}")
print(f"Quality Score:   {((total_rows - critical) / total_rows * 100):.2f}%")

print("\n✅ Profiling complete for title.ratings.tsv")
print("\n📸 TAKE SCREENSHOT FOR YOUR DOCUMENTATION!")

📊 DETAILED PROFILE: title.ratings.tsv

📌 Total Rows: 1,605,649
📌 Total Columns: 3
📌 Columns: ['tconst', 'averageRating', 'numVotes']

📋 SCHEMA
root
 |-- tconst: string (nullable = true)
 |-- averageRating: double (nullable = true)
 |-- numVotes: integer (nullable = true)


📋 COLUMN-WISE PROFILING

🔹 Column: tconst
----------------------------------------
   Non-Null Count : 1,605,649
   Null Count     : 0 (0.00%)
   Distinct Values: 1,605,649
   Min Length     : 9
   Max Length     : 10

🔹 Column: averageRating
----------------------------------------
   Non-Null Count : 1,605,649
   Null Count     : 0 (0.00%)
   Distinct Values: 91
   Min Length     : 3
   Max Length     : 4

🔹 Column: numVotes
----------------------------------------
   Non-Null Count : 1,605,649
   Null Count     : 0 (0.00%)
   Distinct Values: 23,926
   Min Length     : 1
   Max Length     : 7

📌 SAMPLE DATA (First 5 rows)
+---------+-------------+--------+
|tconst   |averageRating|numVotes|
+---------+------------